# Organisation Metrics and Moisture Column

Two analyses:
1. **Rain fraction** — IMERG-derived fraction of rainy pixels per dropsonde circle,
   split by category (Top-Heavy / Bottom-Heavy / Suppressed). Corroborates the
   dynamical classification with independent satellite evidence.
2. **IWV / CWV comparison** — BEACH L4 `iwv_mean` (from dropsonde q profiles) vs
   ERA5 `tcwv` (reanalysis) at each circle location and time, plus scatter vs
   top-heaviness angle.

**Threshold:** 0.5 mm/hr (Tobin et al. 2012, J. Climate)  
**Stratiform proxy:** 0.5–3.0 mm/hr | **Convective proxy:** >3.0 mm/hr (Semie & Bony 2020)

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

from scripts.organisation_metrics import build_circle_metrics

COLOR_TH  = '#d94832'   # Top-Heavy  — red
COLOR_BH  = '#2b83c6'   # Bottom-Heavy — blue
COLOR_SUP = '#6d6d6d'   # Suppressed — grey

CATEGORY_ORDER  = ['Top-Heavy', 'Bottom-Heavy', 'Suppressed']
CATEGORY_COLORS = {'Top-Heavy': COLOR_TH, 'Bottom-Heavy': COLOR_BH, 'Suppressed': COLOR_SUP}

## 2. Load data

In [ ]:
ds_sonde = xr.open_zarr('/g/data/k10/zr7147/ORCESTRA_dropsondes_categorized.zarr')
ds_imerg = xr.open_dataset('/g/data/k10/zr7147/ORCESTRA_IMERG_Combined_Cropped.nc')
era5     = xr.open_dataset('/g/data/k10/zr7147/ERA5/era5_single_levels.nc')

print(f'Circles: {len(ds_sonde.circle)}')
print(f'IMERG timesteps: {len(ds_imerg.time)}')
print(f'ERA5 timesteps: {len(era5.valid_time)}')

## 3. Compute metrics for all circles

In [ ]:
df = build_circle_metrics(ds_sonde, ds_imerg, era5)
print(f'Circles with valid rain fraction: {df["rain_fraction"].notna().sum()} / {len(df)}')
print(f'Category counts:\n{df["category"].value_counts()}')
df.head()

In [ ]:
# Summary statistics by category
df[df['category'].isin(CATEGORY_ORDER)].groupby('category')[
    ['rain_fraction', 'stratiform_proxy_frac', 'convective_proxy_frac', 'iwv_mean', 'era5_tcwv']
].agg(['mean', 'median', 'std']).round(3)

## 4. Rain fraction by category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)

metrics = [
    ('rain_fraction',          'Rain Area Fraction\n(precip > 0.5 mm/hr)',  (0, 0.6)),
    ('stratiform_proxy_frac',  'Stratiform-Proxy Fraction\n(0.5–3.0 mm/hr)', (0, 0.5)),
    ('convective_proxy_frac',  'Convective-Proxy Fraction\n(> 3.0 mm/hr)',   (0, 0.15)),
]

for ax, (col, ylabel, ylim) in zip(axes, metrics):
    data_by_cat = [
        df.loc[df['category'] == cat, col].dropna().values
        for cat in CATEGORY_ORDER
    ]
    n_by_cat = [len(d) for d in data_by_cat]
    colors = [CATEGORY_COLORS[c] for c in CATEGORY_ORDER]

    bp = ax.boxplot(
        data_by_cat,
        patch_artist=True,
        widths=0.45,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
        flierprops=dict(marker='o', markersize=3, alpha=0.5),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for flier, color in zip(bp['fliers'], colors):
        flier.set_markerfacecolor(color)

    # Mann-Whitney U test: TH vs BH
    th_data = data_by_cat[CATEGORY_ORDER.index('Top-Heavy')]
    bh_data = data_by_cat[CATEGORY_ORDER.index('Bottom-Heavy')]
    if len(th_data) >= 3 and len(bh_data) >= 3:
        _, pval = stats.mannwhitneyu(th_data, bh_data, alternative='two-sided')
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        ax.text(0.97, 0.97, f'TH vs BH: {sig}\n(p={pval:.3f})',
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='0.7'))

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([f'{c}\n(n={n})' for c, n in zip(CATEGORY_ORDER, n_by_cat)], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('IMERG Precipitation Metrics by Dropsonde Circle Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/organisation_rain_fraction_by_category.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. IWV/CWV: BEACH L4 vs ERA5 side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: scatter BEACH iwv_mean vs ERA5 tcwv ---
ax = axes[0]
for cat in CATEGORY_ORDER:
    sub = df[df['category'] == cat]
    ax.scatter(sub['era5_tcwv'], sub['iwv_mean'],
               color=CATEGORY_COLORS[cat], label=cat, s=40, alpha=0.75, edgecolors='white', linewidths=0.4)

# 1:1 line
lim = (30, 70)
ax.plot(lim, lim, 'k--', lw=1.2, alpha=0.6, label='1:1 line')

# Regression
valid = df[df['iwv_mean'].notna() & df['era5_tcwv'].notna()]
slope, intercept, r, pval, _ = stats.linregress(valid['era5_tcwv'], valid['iwv_mean'])
xx = np.linspace(*lim, 50)
ax.plot(xx, slope * xx + intercept, color='0.3', lw=1.5, ls='-',
        label=f'OLS  r={r:.2f}')

ax.set_xlabel('ERA5 TCWV  (kg m$^{-2}$)', fontsize=11)
ax.set_ylabel('BEACH L4 IWV  (kg m$^{-2}$)', fontsize=11)
ax.set_xlim(*lim)
ax.set_ylim(*lim)
ax.set_title('BEACH dropsonde IWV vs ERA5 TCWV', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.85)
ax.grid(alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.text(0.04, 0.95,
        f'r = {r:.2f}  (p={pval:.3f})\nslope = {slope:.2f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='0.7'))

# --- Right: IWV distributions by category (violin) ---
ax2 = axes[1]
data_by_cat = [
    df.loc[df['category'] == cat, 'iwv_mean'].dropna().values
    for cat in CATEGORY_ORDER
]
colors = [CATEGORY_COLORS[c] for c in CATEGORY_ORDER]

parts = ax2.violinplot(data_by_cat, positions=[1, 2, 3], showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.65)
parts['cmedians'].set_colors('black')
parts['cmedians'].set_linewidths(2)

# Overlay raw points
for pos, (cat, data) in enumerate(zip(CATEGORY_ORDER, data_by_cat), start=1):
    jitter = np.random.default_rng(42).uniform(-0.08, 0.08, len(data))
    ax2.scatter(pos + jitter, data, color=CATEGORY_COLORS[cat],
                s=18, alpha=0.6, zorder=3, edgecolors='white', linewidths=0.3)

# Mann-Whitney TH vs BH
th_iwv = data_by_cat[CATEGORY_ORDER.index('Top-Heavy')]
bh_iwv = data_by_cat[CATEGORY_ORDER.index('Bottom-Heavy')]
if len(th_iwv) >= 3 and len(bh_iwv) >= 3:
    _, pval2 = stats.mannwhitneyu(th_iwv, bh_iwv, alternative='two-sided')
    sig2 = '***' if pval2 < 0.001 else ('**' if pval2 < 0.01 else ('*' if pval2 < 0.05 else 'ns'))
    ax2.text(0.97, 0.97, f'TH vs BH: {sig2}\n(p={pval2:.3f})',
             transform=ax2.transAxes, ha='right', va='top', fontsize=8,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='0.7'))

ax2.axhline(48, color='0.4', ls='--', lw=1.2, alpha=0.7, label='Deep convection threshold (~48 mm)')
n_by_cat = [len(d) for d in data_by_cat]
ax2.set_xticks([1, 2, 3])
ax2.set_xticklabels([f'{c}\n(n={n})' for c, n in zip(CATEGORY_ORDER, n_by_cat)], fontsize=10)
ax2.set_ylabel('BEACH L4 IWV  (kg m$^{-2}$)', fontsize=11)
ax2.set_title('IWV Distribution by Category', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9, framealpha=0.85)
ax2.grid(axis='y', alpha=0.25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Moisture Column (IWV / CWV) — BEACH L4 vs ERA5', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/iwv_cwv_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. IWV vs top-heaviness angle (the upstream predictor)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: IWV vs top_heaviness_angle ---
ax = axes[0]
for cat in CATEGORY_ORDER:
    sub = df[df['category'] == cat]
    ax.scatter(sub['iwv_mean'], sub['top_heaviness_angle'],
               color=CATEGORY_COLORS[cat], label=cat, s=45, alpha=0.8,
               edgecolors='white', linewidths=0.4)

# Regression on all circles
valid = df[df['iwv_mean'].notna() & df['top_heaviness_angle'].notna()]
slope, intercept, r, pval, _ = stats.linregress(valid['iwv_mean'], valid['top_heaviness_angle'])
xx = np.linspace(valid['iwv_mean'].min(), valid['iwv_mean'].max(), 50)
ax.plot(xx, slope * xx + intercept, 'k-', lw=1.5, label=f'OLS  r={r:.2f}')
ax.axvline(48, color='0.45', ls='--', lw=1.1, alpha=0.7, label='~48 mm threshold')

ax.set_xlabel('BEACH L4 IWV  (kg m$^{-2}$)', fontsize=11)
ax.set_ylabel('Top-heaviness angle (°)', fontsize=11)
ax.set_title('Moisture Column vs Top-heaviness', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.85)
ax.grid(alpha=0.25)
ax.text(0.04, 0.95, f'r = {r:.2f}  (p={pval:.3f})',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='0.7'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: IWV vs rain fraction ---
ax2 = axes[1]
for cat in CATEGORY_ORDER:
    sub = df[(df['category'] == cat) & df['rain_fraction'].notna()]
    ax2.scatter(sub['iwv_mean'], sub['rain_fraction'],
                color=CATEGORY_COLORS[cat], label=cat, s=45, alpha=0.8,
                edgecolors='white', linewidths=0.4)

valid2 = df[df['iwv_mean'].notna() & df['rain_fraction'].notna()]
if len(valid2) >= 5:
    slope2, intercept2, r2, pval2, _ = stats.linregress(valid2['iwv_mean'], valid2['rain_fraction'])
    xx2 = np.linspace(valid2['iwv_mean'].min(), valid2['iwv_mean'].max(), 50)
    ax2.plot(xx2, slope2 * xx2 + intercept2, 'k-', lw=1.5, label=f'OLS  r={r2:.2f}')
    ax2.text(0.04, 0.95, f'r = {r2:.2f}  (p={pval2:.3f})',
             transform=ax2.transAxes, ha='left', va='top', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor='0.7'))

ax2.axvline(48, color='0.45', ls='--', lw=1.1, alpha=0.7, label='~48 mm threshold')
ax2.set_xlabel('BEACH L4 IWV  (kg m$^{-2}$)', fontsize=11)
ax2.set_ylabel('Rain Area Fraction  (> 0.5 mm/hr)', fontsize=11)
ax2.set_title('Moisture Column vs Rain Fraction', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9, framealpha=0.85)
ax2.grid(alpha=0.25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('IWV as Upstream Predictor: Heaviness and Organisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/iwv_vs_heaviness_and_rain_fraction.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Summary: the full causal chain in one table

In [ ]:
summary = (
    df[df['category'].isin(CATEGORY_ORDER)]
    .groupby('category', observed=True)
    .agg(
        n               = ('circle',                'count'),
        iwv_median      = ('iwv_mean',              'median'),
        era5_tcwv_med   = ('era5_tcwv',             'median'),
        rain_frac_med   = ('rain_fraction',         'median'),
        strat_frac_med  = ('stratiform_proxy_frac', 'median'),
        conv_frac_med   = ('convective_proxy_frac', 'median'),
        th_angle_med    = ('top_heaviness_angle',   'median'),
    )
    .reindex(CATEGORY_ORDER)
    .round(3)
)
summary.columns = ['N', 'IWV median', 'ERA5 TCWV median',
                   'Rain fraction', 'Strat proxy', 'Conv proxy', 'TH angle']
print(summary.to_string())